In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
ANALYTICS_DIR = SCRIPT_DIR.parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

OUTPUT_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_03"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Função de exportação
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))

from utils.export_utils import exportar_csv


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# GOLD 03 - DIVERSIDADE DE COR/RAÇA/ETNIA
# ---------------------------------------------------------------------

caminho_gold_03 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_03_diversidade_genero"

arquivos_gold_03 = [
    str(arquivo) for arquivo in caminho_gold_03.glob("part-*.csv")
]

if not arquivos_gold_03:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_03}"
    )

df_genero = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_03)
)


# ---------------------------------------------------------------------
# TOTAL REAL DE RESPONDENTES POR EDIÇÃO
"""
O total de respondentes é recuperado pela dimensão de nível, em vez de somar diretamente as marcações de cor/raça/etnia. A validação posterior mostra que essa dimensão possui mais marcações do que respondentes, portanto suas contagens não representam um total exclusivo de pessoas.
"""
# ---------------------------------------------------------------------

df_nivel = (
    df_genero
    .filter(F.col("variavel") == "nivel")
    .groupBy("edicao", "genero", "valor")
    .agg(F.sum("contagem").alias("contagem"))
)

total_respondentes_edicao = (
    df_nivel
    .groupBy("edicao")
    .agg(F.sum("contagem").alias("total_respondentes"))
)


# ---------------------------------------------------------------------
# BASE DE COR/RAÇA/ETNIA
# ---------------------------------------------------------------------

df_cor_raca_raw = df_genero.filter(F.col("variavel") == "cor_raca_etnia")

df_cor_raca = (
    df_cor_raca_raw
    .groupBy("edicao", "genero", "valor")
    .agg(F.sum("contagem").alias("contagem"))
)


# ---------------------------------------------------------------------
# CATEGORIAS EXISTENTES POR EDIÇÃO
"""
A inspeção confirma que as mesmas sete categorias aparecem nas três edições. Essa estabilidade de nomenclatura permite avaliar a continuidade histórica antes de aplicar critérios adicionais de amostra e relevância analítica.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("1. CATEGORIAS DE COR/RAÇA/ETNIA EXISTENTES POR EDIÇÃO")
print("=" * 100)

(
    df_cor_raca
    .select("edicao", F.col("valor").alias("cor_raca_etnia"))
    .distinct()
    .orderBy("edicao", "cor_raca_etnia")
    .show(100, truncate=False)
)


# ---------------------------------------------------------------------
# VALIDAÇÃO DA DIMENSÃO COR/RAÇA/ETNIA
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("2. VALIDAÇÃO DA DIMENSÃO COR/RAÇA/ETNIA")
print("=" * 100)

total_marcacoes_cor_raca = (
    df_cor_raca
    .groupBy("edicao")
    .agg(F.sum("contagem").alias("total_marcacoes"))
)

validacao_cor_raca = (
    total_marcacoes_cor_raca
    .join(total_respondentes_edicao, on="edicao", how="left")
    .withColumn("diferenca", F.col("total_marcacoes") - F.col("total_respondentes"))
    .withColumn(
        "marcacoes_por_respondente",
        F.round(F.col("total_marcacoes") / F.col("total_respondentes"), 2)
    )
    .orderBy("edicao")
)

validacao_cor_raca.show(20, truncate=False)
"""
Os outputs mostram 1,37 marcação por respondente em 2023-2024 e 2024-2025 e 1,40 em 2025-2026. Como o total de marcações supera o total de respondentes em todas as edições, a dimensão não deve ser interpretada como categorias mutuamente exclusivas; por isso, os percentuais de presença podem somar mais de 100%.
"""


# ---------------------------------------------------------------------
# COMPOSIÇÃO DE COR/RAÇA/ETNIA POR EDIÇÃO
"""
A presença de cada categoria é calculada sobre o total real de respondentes da edição. O indicador representa a proporção de pessoas associadas a cada categoria e não uma distribuição que precise totalizar 100%, coerentemente com a validação de múltiplas marcações.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("3. COMPOSIÇÃO DE COR/RAÇA/ETNIA POR EDIÇÃO")
print("=" * 100)

total_cor_raca = (
    df_cor_raca
    .groupBy("edicao", "valor")
    .agg(F.sum("contagem").alias("respondentes"))
    .withColumnRenamed("valor", "cor_raca_etnia")
)

composicao_cor_raca = (
    total_cor_raca
    .join(total_respondentes_edicao, on="edicao", how="left")
    .withColumn(
        "pct_respondentes",
        F.round(F.col("respondentes") / F.col("total_respondentes") * 100, 1)
    )
    .orderBy("edicao", F.desc("respondentes"))
)

composicao_cor_raca.show(100, truncate=False)
"""
No histórico, Branca concentra a maior presença, com 88,5% em 2023-2024 e 90,9% em 2025-2026. Parda permanece próxima de um terço dos respondentes, enquanto Preta passa de 10,0% para 8,7% no mesmo período.
"""


# ---------------------------------------------------------------------
# TOTAL DE RESPONDENTES POR EDIÇÃO
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("4. TOTAL DE RESPONDENTES POR EDIÇÃO")
print("=" * 100)

total_respondentes_edicao.orderBy("edicao").show(20, truncate=False)


# ---------------------------------------------------------------------
# REPRESENTATIVIDADE DE GÊNERO POR COR/RAÇA/ETNIA
"""
Neste recorte, o denominador muda: a composição de gênero é calculada dentro de cada categoria de cor/raça/etnia. Assim, o percentual responde qual é a participação de cada gênero entre as pessoas associadas àquela categoria, sem usar o total geral da edição.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("5. REPRESENTATIVIDADE DE GÊNERO POR COR/RAÇA/ETNIA")
print("=" * 100)

janela_cor_raca = Window.partitionBy("edicao", "valor")

genero_por_cor_raca = (
    df_cor_raca
    .withColumn("total_cor_raca", F.sum("contagem").over(janela_cor_raca))
    .withColumn(
        "pct_no_grupo",
        F.round(F.col("contagem") / F.col("total_cor_raca") * 100, 1)
    )
    .select(
        "edicao",
        F.col("valor").alias("cor_raca_etnia"),
        "genero",
        "contagem",
        "total_cor_raca",
        "pct_no_grupo"
    )
    .orderBy("edicao", F.desc("total_cor_raca"), F.desc("contagem"))
)

genero_por_cor_raca.show(200, truncate=False)


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR COR/RAÇA/ETNIA
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("6. PARTICIPAÇÃO FEMININA POR COR/RAÇA/ETNIA")
print("=" * 100)

participacao_feminina_cor_raca = (
    genero_por_cor_raca
    .filter(F.col("genero") == "Feminino")
    .select(
        "edicao",
        "cor_raca_etnia",
        F.col("contagem").alias("mulheres"),
        "total_cor_raca",
        F.col("pct_no_grupo").alias("pct_feminino")
    )
    .orderBy("edicao", F.desc("total_cor_raca"))
)

participacao_feminina_cor_raca.show(100, truncate=False)
"""
A leitura isolada de categorias com amostras muito pequenas pode gerar percentuais instáveis. Por isso, os resultados desta etapa são usados junto com a checagem de comparabilidade antes de construir a série histórica.
"""


# ---------------------------------------------------------------------
# COMPARABILIDADE DAS CATEGORIAS ENTRE AS EDIÇÕES
"""
A comparabilidade verifica simultaneamente presença nas três edições e tamanho de amostra. Embora todas as categorias apareçam nos três anos, os volumes variam bastante: Branca, Parda, Preta e Amarela mantêm amostras superiores às categorias Indígena e Outra.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("7. COMPARABILIDADE DAS CATEGORIAS ENTRE AS EDIÇÕES")
print("=" * 100)

comparabilidade_cor_raca = (
    composicao_cor_raca
    .groupBy("cor_raca_etnia")
    .agg(
        F.countDistinct("edicao").alias("qtd_edicoes"),
        F.min("respondentes").alias("menor_amostra"),
        F.max("respondentes").alias("maior_amostra")
    )
    .orderBy(F.desc("qtd_edicoes"), F.desc("maior_amostra"))
)

comparabilidade_cor_raca.show(100, truncate=False)


# ---------------------------------------------------------------------
# CATEGORIAS ELEGÍVEIS PARA ANÁLISE HISTÓRICA
"""
Para a comparação histórica, são mantidas apenas categorias presentes nas três edições e com pelo menos 20 respondentes em todas elas. "Prefiro não informar" e "Outra" também são retiradas por não representarem categorias raciais específicas para o objetivo da análise.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("8. CATEGORIAS ELEGÍVEIS PARA ANÁLISE HISTÓRICA")
print("=" * 100)

categorias_elegiveis = (
    comparabilidade_cor_raca
    .filter(
        (F.col("qtd_edicoes") == 3)
        & (F.col("menor_amostra") >= 20)
        & (~F.col("cor_raca_etnia").isin("Prefiro não informar", "Outra"))
    )
)

categorias_elegiveis.show(100, truncate=False)
"""
Com os critérios definidos, permanecem Branca, Parda, Preta e Amarela. A categoria Indígena não atinge o corte mínimo de amostra, enquanto "Outra" também possui baixa amostra e "Prefiro não informar" é excluída pelo critério analítico.
"""


# ---------------------------------------------------------------------
# EVOLUÇÃO DA PRESENÇA DE COR/RAÇA/ETNIA
"""
A série histórica passa a considerar somente as categorias elegíveis, evitando que grupos com amostras muito pequenas influenciem comparações entre as edições.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("9. EVOLUÇÃO DA PRESENÇA DE COR/RAÇA/ETNIA")
print("=" * 100)

composicao_cor_raca_historica = (
    composicao_cor_raca
    .join(categorias_elegiveis.select("cor_raca_etnia"), on="cor_raca_etnia", how="inner")
    .select(
        "edicao",
        "cor_raca_etnia",
        "respondentes",
        "total_respondentes",
        "pct_respondentes"
    )
    .orderBy("cor_raca_etnia", "edicao")
)

composicao_cor_raca_historica.show(100, truncate=False)


# ---------------------------------------------------------------------
# VARIAÇÃO ENTRE 2023-2024 E 2025-2026
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("10. VARIAÇÃO DA PRESENÇA RACIAL | 2023-2024 x 2025-2026")
print("=" * 100)

variacao_composicao_cor_raca = (
    composicao_cor_raca_historica
    .groupBy("cor_raca_etnia")
    .agg(
        F.max(
            F.when(F.col("edicao") == "2023-2024", F.col("pct_respondentes"))
        ).alias("pct_2023_2024"),
        F.max(
            F.when(F.col("edicao") == "2025-2026", F.col("pct_respondentes"))
        ).alias("pct_2025_2026")
    )
    .withColumn(
        "variacao_pp",
        F.round(F.col("pct_2025_2026") - F.col("pct_2023_2024"), 1)
    )
    .orderBy(F.desc("variacao_pp"))
)

variacao_composicao_cor_raca.show(100, truncate=False)
"""
Entre 2023-2024 e 2025-2026, a presença de Branca aumenta 2,4 pontos percentuais, Amarela 0,9 p.p. e Parda 0,5 p.p. Já Preta apresenta redução de 1,3 p.p. no período.
"""


# ---------------------------------------------------------------------
# PARTICIPAÇÃO FEMININA POR COR/RAÇA/ETNIA - HISTÓRICO
"""
O recorte cruza as duas dimensões de diversidade mantendo os mesmos critérios de comparabilidade racial. O objetivo é observar se a participação feminina evolui de maneira semelhante ou diferente entre as categorias elegíveis.
"""
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("11. PARTICIPAÇÃO FEMININA POR COR/RAÇA/ETNIA - HISTÓRICO")
print("=" * 100)

participacao_feminina_cor_raca_historica = (
    participacao_feminina_cor_raca
    .join(categorias_elegiveis.select("cor_raca_etnia"), on="cor_raca_etnia", how="inner")
    .select(
        "edicao",
        "cor_raca_etnia",
        "mulheres",
        "total_cor_raca",
        "pct_feminino"
    )
    .orderBy("cor_raca_etnia", "edicao")
)

participacao_feminina_cor_raca_historica.show(100, truncate=False)


# ---------------------------------------------------------------------
# VARIAÇÃO DA PARTICIPAÇÃO FEMININA
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("12. VARIAÇÃO DA PARTICIPAÇÃO FEMININA | 2023-2024 x 2025-2026")
print("=" * 100)

variacao_participacao_feminina = (
    participacao_feminina_cor_raca_historica
    .groupBy("cor_raca_etnia")
    .agg(
        F.max(
            F.when(F.col("edicao") == "2023-2024", F.col("pct_feminino"))
        ).alias("pct_feminino_2023_2024"),
        F.max(
            F.when(F.col("edicao") == "2025-2026", F.col("pct_feminino"))
        ).alias("pct_feminino_2025_2026")
    )
    .withColumn(
        "variacao_pp",
        F.round(F.col("pct_feminino_2025_2026") - F.col("pct_feminino_2023_2024"), 1)
    )
    .orderBy(F.desc("variacao_pp"))
)

variacao_participacao_feminina.show(100, truncate=False)
"""
A participação feminina diminui nas quatro categorias elegíveis entre 2023-2024 e 2025-2026. As variações são de -1,7 p.p. em Parda, -2,5 p.p. em Branca, -3,1 p.p. em Amarela e -3,4 p.p. em Preta.
"""


# ---------------------------------------------------------------------
# RETRATO ATUAL DA PARTICIPAÇÃO FEMININA POR COR/RAÇA/ETNIA
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("13. PARTICIPAÇÃO FEMININA POR COR/RAÇA/ETNIA | 2025-2026")
print("=" * 100)

participacao_feminina_cor_raca_atual = (
    participacao_feminina_cor_raca_historica
    .filter(F.col("edicao") == "2025-2026")
    .orderBy(F.desc("pct_feminino"))
)

participacao_feminina_cor_raca_atual.show(100, truncate=False)
"""
No retrato de 2025-2026, Amarela apresenta a maior participação feminina entre as categorias elegíveis, com 24,8%, seguida por Preta com 23,5%, Branca com 22,0% e Parda com 21,5%.
"""


# ---------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS
# ---------------------------------------------------------------------

exportar_csv(
    composicao_cor_raca_historica,
    OUTPUT_DIR,
    "composicao_cor_raca_historica.csv"
)

exportar_csv(
    participacao_feminina_cor_raca_historica,
    OUTPUT_DIR,
    "participacao_feminina_cor_raca_historica.csv"
)